# NLP Lab 4 Task

### **Instructions for Students:**
1. All data preparation, vocabularies, and boilerplate loader functions are pre-written.
2. Do **NOT** copy-paste standard models from the lab demo. Read the specific architectural or algorithmic twists for each task carefully.
3. You must replace all `raise NotImplementedError()` blocks with your code implementation.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import gensim.downloader as api
import numpy as np
import random
import urllib.request
import re

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load Pre-trained Google Word2Vec (300-dim)
print("Loading Google Word2Vec Model...")
w2v_google = api.load('word2vec-google-news-300')

def build_embedding_matrix(corpus, w2v_model):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    embedding_dim = w2v_model.vector_size
    weights = [np.zeros(embedding_dim), np.random.randn(embedding_dim) * 0.1]
    
    for sentence in corpus:
        for word in sentence:
            if word not in vocab:
                vocab[word] = len(vocab)
                if word in w2v_model:
                    weights.append(w2v_model[word])
                else:
                    weights.append(np.random.randn(embedding_dim) * 0.1)
                    
    return vocab, torch.tensor(np.array(weights), dtype=torch.float32)

# Text Classification via Mean-Pooled BiLSTM
### **The Twist / Challenge:**
In the lab, you used the last hidden state ($h_n$) for classification. However, the last hidden state can create an information bottleneck.For multi-class emotion classification (4 classes: 0: Joy, 1: Anger, 2: Sadness, 3: Fear), you must build a BiLSTM that applies Mean Pooling over all time-steps of the LSTM output sequence ($\text{out} \in \mathbb{B} \times \text{seq\_len} \times 2h$), ignoring <PAD> tokens using a binary mask.

In [ ]:
#Data Setup (Provided)
emotion_dataset = [
    ("i am so happy and excited today", 0), ("fantastic news i love it", 0),
    ("this makes me so angry and furious", 1), ("horrible service disgusting attitude", 1),
    ("i feel deeply sad and heartbroken", 2), ("crying all night extremely depressed", 2),
    ("scared of the dark very frightened", 3), ("terrified about the upcoming exam", 3)
]

corpus_t1 = [s.lower().split() for s, _ in emotion_dataset]
vocab_t1, embed_matrix_t1 = build_embedding_matrix(corpus_t1, w2v_google)

def encode_t1(sent, vocab, max_len=8):
    tokens = [vocab.get(w, vocab["<UNK>"]) for w in sent.lower().split()]
    return tokens[:max_len] + [vocab["<PAD>"]] * max(0, max_len - len(tokens))

X_t1 = torch.tensor([encode_t1(s, vocab_t1) for s, _ in emotion_dataset], dtype=torch.long).to(device)
y_t1 = torch.tensor([label for _, label in emotion_dataset], dtype=torch.long).to(device)

In [ ]:
class MeanPooledBiLSTMClassifier(nn.Module):
    def __init__(self, embed_matrix, hidden_dim, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embed_matrix, freeze=False, padding_idx=0)
        self.hidden_dim = hidden_dim
        # TODO: Initialize bidirectional LSTM (batch_first=True)
        # TODO: Initialize Linear output layer mapping from pooled representation to num_classes
        raise NotImplementedError("Student Task: Initialize model layers")

    def forward(self, x):
        """
        Input x shape: (batch_size, seq_len)
        """
        # TODO 1: Extract embeddings
        # TODO 2: Pass through BiLSTM to get sequence output of shape (batch, seq_len, hidden_dim * 2)
        # TODO 3: Create a boolean mask where padding tokens (index 0) are 0, others are 1.
        #         Mask shape: (batch, seq_len, 1)
        # TODO 4: Apply mask to zero-out padded time-steps, compute mean over seq_len dimension,
        #         and pass the pooled vector through linear layer to return logits.
        raise NotImplementedError("Student Task: Implement Mean Pooling with Padding Masking")

# Instantiate and test forward pass shape
model_t1 = MeanPooledBiLSTMClassifier(embed_matrix_t1, hidden_dim=16, num_classes=4).to(device)
out_t1 = model_t1(X_t1)
print(f"Task 1 Output Logits Shape (Expected: [8, 4]): {out_t1.shape}")